# External Framework Validation — SynthEval (Bank Marketing)

Runs in the isolated `tese_eval` conda environment (syntheval, Python 3.10).
Mirrors the Dataset 1 SynthEval notebook, adapted to the Bank Marketing feature
space. SMOTE is excluded, and there is no Variant A/B (no ordinal columns).

## 0. Configuration and paths

In [1]:
from pathlib import Path
import warnings
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
warnings.filterwarnings("ignore")

RANDOM_STATE = 123

# Folder with the CSV files, relative to this notebook (adjust if needed)
DATA_DIR = Path("data")

assert DATA_DIR.exists(), f"Folder not found: {DATA_DIR}"
print("Data folder:", DATA_DIR)
from syntheval import SynthEval

Data folder: data


## 1. Canonical column definitions (Bank Marketing)

In [2]:
TARGET = "y"

FEATURES = ['age','job','marital','education','default','housing','loan','contact',
            'month','day_of_week','campaign','previous','poutcome','emp.var.rate',
            'cons.price.idx','cons.conf.idx','euribor3m','nr.employed','was_contacted']

# Categorical columns: the 11 nominal categoricals (incl. the binary flag).
# Bank Marketing has NO ordinal columns, so there is a single, unambiguous
# categorical treatment (no Variant A/B, unlike the PAY_X columns of Dataset 1).
CAT_COLS = ['job','marital','education','default','housing','loan','contact',
            'month','day_of_week','poutcome','was_contacted']
NUM_COLS = [c for c in FEATURES if c not in CAT_COLS]

assert len(FEATURES) == 19, f"FEATURES should have 19, got {len(FEATURES)}"
assert TARGET not in FEATURES, "y must not be in FEATURES"
assert len(CAT_COLS) == 11 and len(NUM_COLS) == 8
print("FEATURES:", len(FEATURES), "| CAT:", len(CAT_COLS), "| NUM:", len(NUM_COLS))

FEATURES: 19 | CAT: 11 | NUM: 8


## 2. Helper: align any dataframe to the 19 features

In [3]:
def to_features(df, name):
    d = df.copy()
    if TARGET in d.columns:
        d = d.drop(columns=[TARGET])
    d = d[FEATURES]                       # enforce order; errors if any missing
    d[NUM_COLS] = d[NUM_COLS].astype("float64")
    assert list(d.columns) == FEATURES
    print(f"{name:<16} -> {d.shape}")
    return d

## 3. Load the synthetic datasets and the real minority reference

In [4]:
real_min  = pd.read_csv(DATA_DIR / "bank_real_minority_train.csv")
synth_ctg = pd.read_csv(DATA_DIR / "bank_synthetic_ctgan_final.csv")
synth_ctb = pd.read_csv(DATA_DIR / "bank_synthetic_ctab_final.csv")

real_min_p  = to_features(real_min,  "real_minority")
synth_ctg_p = to_features(synth_ctg, "ctgan")
synth_ctb_p = to_features(synth_ctb, "ctabganplus")

for d in (synth_ctg_p, synth_ctb_p):
    assert list(d.columns) == list(real_min_p.columns)
print("\nAll columns aligned.")

real_minority    -> (3248, 19)
ctgan            -> (22335, 19)
ctabganplus      -> (22335, 19)

All columns aligned.


## 4. Reconstruct the canonical split → holdout
The holdout (real minority never seen by the GANs) lets SynthEval measure
train-vs-holdout privacy losses.

In [5]:
df_raw = pd.read_csv(DATA_DIR / "bank-additional-full.csv", sep=';')

df_c = df_raw.drop(columns=['duration'])
df_c['was_contacted'] = (df_c['pdays'] != 999).astype(int)
df_c = df_c.drop(columns=['pdays'])
df_c[TARGET] = (df_c['y'] == 'yes').astype(int)
df_c['default'] = df_c['default'].replace({'yes': 'unknown'})

X = df_c.drop(columns=TARGET)
y = df_c[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=RANDOM_STATE)

# verify reproduction against the stored training partition
real_full = pd.read_csv(DATA_DIR / "bank_real_train_full.csv")
train_recon = X_train.copy()
train_recon[TARGET] = y_train
train_recon = train_recon[real_full.columns].reset_index(drop=True)
assert train_recon.equals(real_full.reset_index(drop=True)), \
    "Reconstructed split != bank_real_train_full.csv"
print("Split reproduced bit for bit. OK.")

# Holdout = real minority in the test set (y==1), never seen by the GANs
holdout_raw = X_test[y_test == 1].reset_index(drop=True)
holdout = to_features(holdout_raw, "holdout")
assert list(holdout.columns) == FEATURES

Split reproduced bit for bit. OK.
holdout          -> (1392, 19)


## 5. Fidelity metrics (no holdout needed)
Explicit metric selection (mirrors Dataset 1): KS test, correlation difference,
and propensity MSE (pMSE).

In [6]:
ev_fid = SynthEval(real_min_p, cat_cols=CAT_COLS)

def fidelity(synth, name):
    print(f"\n{'='*55}\n{name}\n{'='*55}")
    return ev_fid.evaluate(synth, ks_test={}, corr_diff={}, p_mse={})

res_ctgan = fidelity(synth_ctg_p, "CTGAN")
res_ctab  = fidelity(synth_ctb_p, "CTAB-GAN+")

Rich console is not supported in this environment. Defaulting to ascii console.

CTGAN
SynthEval: synthetic data read successfully


Syntheval: p_mse: 100%|██████████| 3/3 [00:35<00:00, 11.72s/it]    


SynthEval results
Utility metric description                      value    error
+----------------------------------------------------------------+
| General Kolmogorov–Smirnov Statistic       :  0.1094   0.0148  |
|   -> Avg. Kolmogorov–Smirnov dist.         :  0.1396   0.0218  |
|   -> Avg. Total Variation Distance         :  0.0873   0.0181  |
| Fraction of Significant KS Tests           :  0.8421           |
|   -> # of Significant Tests at a=0.05      :  16.0000           |
|   -> Avg. combined p-value                 :  0.0730   0.0530  |
| Mixed correlation matrix difference        :  1.9464           |
| Propensity mean squared error (pMSE)       :  0.0045   0.0001  |
|   -> average pMSE classifier accuracy      :  0.4664   0.0003  |
+----------------------------------------------------------------+


CTAB-GAN+
SynthEval: synthetic data read successfully


Syntheval: p_mse: 100%|██████████| 3/3 [00:37<00:00, 12.45s/it]    

SynthEval results
Utility metric description                      value    error
+----------------------------------------------------------------+
| General Kolmogorov–Smirnov Statistic       :  0.1457   0.0225  |
|   -> Avg. Kolmogorov–Smirnov dist.         :  0.2225   0.0260  |
|   -> Avg. Total Variation Distance         :  0.0899   0.0221  |
| Fraction of Significant KS Tests           :  1.0000           |
|   -> # of Significant Tests at a=0.05      :  19.0000           |
|   -> Avg. combined p-value                 :  0.0006   0.0001  |
| Mixed correlation matrix difference        :  2.7660           |
| Propensity mean squared error (pMSE)       :  0.0103   0.0002  |
|   -> average pMSE classifier accuracy      :  0.5110   0.0028  |
+----------------------------------------------------------------+



In [7]:
def tag(res, model):
    d = res.copy(); d.insert(0, "model", model); return d

fidelity_all = pd.concat([tag(res_ctgan, "CTGAN"),
                          tag(res_ctab,  "CTAB-GAN+")], ignore_index=True)

print(fidelity_all.pivot(index="model", columns="metric", values="val").round(4))
fidelity_all.to_csv(DATA_DIR / "bank_syntheval_fidelity_results.csv", index=False)
print("\nSaved: bank_syntheval_fidelity_results.csv")

metric     avg_pMSE  corr_mat_diff  frac_ks_sigs  ks_tvd_stat
model                                                       
CTAB-GAN+    0.0103         2.7660        1.0000       0.1457
CTGAN        0.0045         1.9464        0.8421       0.1094

Saved: bank_syntheval_fidelity_results.csv


## 6. Privacy / memorisation metrics
Explicit metrics (mirrors Dataset 1): NNAA, DCR, and epsilon-identifiability
risk. No Variant A/B: Bank Marketing has no ordinal columns.

In [8]:
ev_priv = SynthEval(real_min_p, holdout_dataframe=holdout, cat_cols=CAT_COLS)

def privacy(synth, name):
    print(f"\n{'='*55}\n{name}\n{'='*55}")
    return ev_priv.evaluate(synth, nnaa={}, dcr={}, eps_risk={})

priv_ctgan = privacy(synth_ctg_p, "CTGAN")
priv_ctab  = privacy(synth_ctb_p, "CTAB-GAN+")

Rich console is not supported in this environment. Defaulting to ascii console.

CTGAN
SynthEval: synthetic data read successfully


Syntheval: eps_risk: 100%|██████████| 3/3 [01:19<00:00, 26.42s/it]


SynthEval results
Utility metric description                      value    error
+----------------------------------------------------------------+
| Nearest neighbour adversarial accuracy     :  0.7791   0.0007  |
+----------------------------------------------------------------+

Privacy metric description                      value    error
+----------------------------------------------------------------+
| Privacy loss (diff. in NNAA)               : -0.0211   0.0015  |
| Median distance to closest record          :  2.0974           |
| Epsilon identifiability risk               :  0.2482           |
| Privacy loss (diff. in eps. risk)          :  0.0705           |
+----------------------------------------------------------------+


CTAB-GAN+
SynthEval: synthetic data read successfully


Syntheval: eps_risk: 100%|██████████| 3/3 [01:26<00:00, 28.86s/it]

SynthEval results
Utility metric description                      value    error
+----------------------------------------------------------------+
| Nearest neighbour adversarial accuracy     :  0.7951   0.0009  |
+----------------------------------------------------------------+

Privacy metric description                      value    error
+----------------------------------------------------------------+
| Privacy loss (diff. in NNAA)               : -0.0258   0.0014  |
| Median distance to closest record          :  2.1108           |
| Epsilon identifiability risk               :  0.3110           |
| Privacy loss (diff. in eps. risk)          :  0.1053           |
+----------------------------------------------------------------+



In [9]:
privacy_all = pd.concat([tag(priv_ctgan, "CTGAN"),
                         tag(priv_ctab,  "CTAB-GAN+")], ignore_index=True)

print("metric names:", privacy_all["metric"].unique())
print()
print(privacy_all.pivot(index="model", columns="metric", values="val").round(4))
privacy_all.to_csv(DATA_DIR / "bank_syntheval_privacy_results.csv", index=False)
print("\nSaved: bank_syntheval_privacy_results.csv")

metric names: ['nnaa' 'priv_loss_nnaa' 'median_DCR' 'eps_identif_risk' 'priv_loss_eps']

metric     eps_identif_risk  median_DCR    nnaa  priv_loss_eps  priv_loss_nnaa
model                                                                        
CTAB-GAN+            0.3110      2.1108  0.7951         0.1053         -0.0258
CTGAN                0.2482      2.0974  0.7791         0.0705         -0.0211

Saved: bank_syntheval_privacy_results.csv


---

In [10]:
print("=== FIDELITY ===")
print(fidelity_all.pivot(index="model", columns="metric", values="val").round(4))
print("\n=== PRIVACY / MEMORISATION ===")
print(privacy_all.pivot(index="model", columns="metric", values="val").round(4))

=== FIDELITY ===
metric     avg_pMSE  corr_mat_diff  frac_ks_sigs  ks_tvd_stat
model                                                       
CTAB-GAN+    0.0103         2.7660        1.0000       0.1457
CTGAN        0.0045         1.9464        0.8421       0.1094

=== PRIVACY / MEMORISATION ===
metric     eps_identif_risk  median_DCR    nnaa  priv_loss_eps  priv_loss_nnaa
model                                                                        
CTAB-GAN+            0.3110      2.1108  0.7951         0.1053         -0.0258
CTGAN                0.2482      2.0974  0.7791         0.0705         -0.0211
